# Minimum capital per strategy and portfolio

Everything is read from one YAML config (instruments, strategies, candidate instruments per platform, portfolios, assumptions). Copy `portfolio_capital_config.yaml`, edit it and set `CONFIG_PATH` in the first cell.

**Flow:** config → minimum capital per candidate → cheapest instrument per platform → charts per strategy → charts per portfolio.

**One strategy:** `capital = stop × usd_per_unit / r` — one minimum lot may risk at most `r` of equity.

**A portfolio** (`PORTFOLIO_RULE`):
- `sum` — every strategy can be open at the same time and their *combined* min-lot risk stays ≤ `r`. Conservative, default.
- `max` — each strategy only has to respect `r` on its own (assumes they are rarely open together).

> This notebook does **not** measure any edge. It never reads prices or backtests: stops, ATR% and costs are numbers typed in the YAML. The *Data basis* table shows which strategies are backtested and which are still assumptions.

In [1]:
CONFIG_PATH = "portfolio_capital_config.yaml"   # file next to the notebook; None -> uses CONFIG_YAML defined in a cell
PORTFOLIO_RULE = "sum"                          # "sum" (all open at once) | "max" (each strategy alone)

In [2]:
import math, pathlib
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb, to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from IPython.display import HTML, Markdown, display

# ── theme (same palette as the RSI(2) backtest cell) ──
T = dict(bg="#15171c", panel="#1b1e26", grid="#2a2d37", text="#e8e9ec", muted="#9aa0ab",
         green="#34d399", blue="#5468ff", amber="#f5a524", red="#f87171")
CYCLE = [T["green"], T["blue"], T["amber"], "#a78bfa", "#22d3ee"]

plt.rcParams.update({
    "figure.facecolor": T["bg"], "savefig.facecolor": T["bg"], "axes.facecolor": T["panel"],
    "axes.edgecolor": T["grid"], "axes.labelcolor": T["muted"], "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False, "axes.spines.bottom": False,
    "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 10.5, "axes.titlepad": 10,
    "text.color": T["text"], "xtick.color": T["muted"], "ytick.color": T["text"],
    "xtick.labelsize": 9, "ytick.labelsize": 9.5, "xtick.major.size": 0, "ytick.major.size": 0,
    "grid.color": T["grid"], "grid.linewidth": 0.8, "axes.grid": True, "axes.grid.axis": "x", "axes.axisbelow": True,
    "legend.frameon": False, "legend.labelcolor": T["muted"], "legend.fontsize": 9,
})
if "text.parse_math" in plt.rcParams:      # matplotlib >= 3.6: keep "$1,840 ... $3,333" as plain text
    plt.rcParams["text.parse_math"] = False
    esc = lambda s: s
else:                                      # older versions: escape the dollar sign
    esc = lambda s: s.replace("$", r"\$")

def ink(color):
    """Readable text colour on top of a bar colour."""
    r, g, b = to_rgb(color)
    return T["bg"] if 0.299 * r + 0.587 * g + 0.114 * b > 0.45 else "#ffffff"

# ── config ──
def load_config(path):
    if path:
        p = pathlib.Path(path).expanduser()
        for c in (p, pathlib.Path(str(p) + ".yaml"), pathlib.Path(str(p) + ".yml")):
            if c.is_file():
                return yaml.safe_load(c.read_text()), str(c)
        found = [f.name for f in pathlib.Path.cwd().glob("*.y*ml")]
        raise FileNotFoundError(f"Config '{path}' not found (also tried .yaml/.yml) in {pathlib.Path.cwd()}. YAML files here: {found or 'none'}")
    if "CONFIG_YAML" in globals():
        return yaml.safe_load(CONFIG_YAML), "embedded CONFIG_YAML"
    raise NameError("CONFIG_PATH is None but there is no CONFIG_YAML cell")

cfg, CFG_SRC = load_config(CONFIG_PATH)
S, PL = cfg["settings"], cfg["platforms"]
R = S["r"]
NAN = float("nan")
PCOL = {p: CYCLE[i % len(CYCLE)] for i, p in enumerate(PL)}                 # platform colours
SCOL = {s: CYCLE[i % len(CYCLE)] for i, s in enumerate(cfg["strategies"])}  # strategy colours
assert PORTFOLIO_RULE in ("sum", "max"), "PORTFOLIO_RULE must be 'sum' or 'max'"

def usd(x, d=0):
    return "verify" if x is None or (isinstance(x, float) and math.isnan(x)) else f"${x:,.{d}f}"

# ── core maths (unchanged logic) ──
def inst(plat, sym):
    d = dict(cfg["instruments"][plat][sym])
    d["notional"] = d["usd_per_unit"] * d["price"]
    d["margin"] = d["notional"] * d["margin_pct"]
    return d

def stop_units(strategy, plat, sym):
    st, d = cfg["strategies"][strategy]["stop"], inst(plat, sym)
    if st["type"] == "atr_multiple":
        return round(st["multiple"] * d["atr_pct"] / 100 * d["price"])
    v = st["value"]
    return v[sym] if isinstance(v, dict) else v

def min_capital(strategy, plat, sym, gap_control=False):
    d = inst(plat, sym)
    cap = stop_units(strategy, plat, sym) * d["usd_per_unit"] / R
    gc = cfg["strategies"][strategy].get("gap_control")
    if gap_control and gc:
        cap = max(cap, gc["adverse_move"] * d["usd_per_unit"] / gc["max_equity_pct"])
    return cap

def candidates(strategy, gap_control=False):
    rows = []
    for plat, syms in cfg["strategies"][strategy]["candidates"].items():
        for sym in syms:
            d = inst(plat, sym)
            rows.append(dict(plat=plat, sym=sym, cap=min_capital(strategy, plat, sym, gap_control),
                             stop=stop_units(strategy, plat, sym), margin=d["margin"], notional=d["notional"]))
    return pd.DataFrame(rows).sort_values(["cap", "margin"]).reset_index(drop=True)

def cheapest(strategy, gap_control=False):
    """Cheapest candidate per platform: {platform: (symbol, capital, margin)}"""
    df = candidates(strategy, gap_control)
    return {p: (g.iloc[0].sym, g.iloc[0].cap, g.iloc[0].margin) for p, g in df.groupby("plat", sort=False)}

def scenarios_for(strategy):
    has_gc = "gap_control" in cfg["strategies"][strategy]
    return [s for s in S["scenarios"] if has_gc or not s["gap_control"]]

def scenarios_for_portfolio(pname):
    has_gc = any("gap_control" in cfg["strategies"][s] for s in cfg["portfolios"][pname])
    return has_gc, [s for s in S["scenarios"] if has_gc or not s["gap_control"]]

def portfolio_summary(pname, gap_control):
    """Per platform: each strategy's cheapest instrument, capital as SUM and as MAX, margin."""
    strategies = cfg["portfolios"][pname]
    picks = {p: {} for p in PL}
    for strat in strategies:
        for p, v in cheapest(strat, gap_control).items():
            picks[p][strat] = v
    out = {}
    for p, pk in picks.items():
        if len(pk) < len(strategies):
            continue   # platform cannot run every strategy of this portfolio
        caps = {s: v[1] for s, v in pk.items()}
        driver = max(caps, key=caps.get)
        out[p] = dict(picks=pk, caps=caps, cap_sum=sum(caps.values()), cap_max=caps[driver], driver=driver,
                      margin=sum(v[2] for v in pk.values()))
    return out

def rule_cap(v):
    return v["cap_sum"] if PORTFOLIO_RULE == "sum" else v["cap_max"]

# ── table helper (dark cards) ──
def show(df, title=None, text_cols=1, accents=None, bold_rows=None, cell_css=None, fmt=None):
    sty = df.style.hide(axis="index").format(escape="html")
    if fmt:
        sty = sty.format(fmt, escape="html")
    def css(d):
        out = pd.DataFrame("", index=d.index, columns=d.columns)
        for i in range(len(d)):
            for j in range(d.shape[1]):
                parts = []
                if accents is not None and j == 0:
                    parts.append(f"border-left: 3px solid {accents[i]}")
                if bold_rows is not None:
                    parts.append("font-weight: 600" if bold_rows[i] else f"color: {T['muted']}")
                if cell_css is not None:
                    extra = cell_css(d).iloc[i, j]
                    if extra:
                        parts.append(extra)
                out.iloc[i, j] = "; ".join(parts)
        return out
    sty = sty.apply(css, axis=None)
    styles = [
        dict(selector="", props=[("border-collapse", "separate"), ("border-spacing", "0"), ("background", T["panel"]),
                                 ("color", T["text"]), ("font", "12.5px/1.35 -apple-system, 'Segoe UI', Inter, sans-serif"),
                                 ("border", f"1px solid {T['grid']}"), ("border-radius", "10px"), ("overflow", "hidden"),
                                 ("margin", "6px 0 16px 0")]),
        dict(selector="caption", props=[("caption-side", "top"), ("text-align", "left"), ("color", T["text"]),
                                        ("font-weight", "600"), ("font-size", "13px"), ("padding", "0 0 8px 2px")]),
        dict(selector="th", props=[("background", T["bg"]), ("color", T["muted"]), ("font-weight", "500"), ("font-size", "10.5px"),
                                   ("text-transform", "uppercase"), ("letter-spacing", ".06em"), ("padding", "8px 12px"),
                                   ("text-align", "right"), ("border-bottom", f"1px solid {T['grid']}")]),
        dict(selector="td", props=[("padding", "7px 12px"), ("text-align", "right"), ("border-bottom", f"1px solid {T['grid']}"),
                                   ("font-variant-numeric", "tabular-nums")]),
        dict(selector="tbody tr:last-child td", props=[("border-bottom", "none")]),
        dict(selector="tbody tr:hover td", props=[("background", "#222633")]),
    ]
    sty = sty.set_table_styles(styles)
    left = {c: [dict(selector="", props=[("text-align", "left")])] for c in df.columns[:text_cols]}
    sty = sty.set_table_styles(left, overwrite=False)
    if title:
        sty = sty.set_caption(title)
    display(sty)

def chip(text, color):
    return (f"<span style='background:{color}22;color:{color};border:1px solid {color}55;border-radius:999px;"
            f"padding:2px 10px;font-size:11.5px;font-weight:600;margin-right:6px'>{text}</span>")

n_inst = sum(len(v) for v in cfg["instruments"].values())
display(HTML(
    f"<div style='background:{T['panel']};border:1px solid {T['grid']};border-radius:10px;padding:10px 14px;"
    f"color:{T['muted']};font:12.5px -apple-system,Segoe UI,Inter,sans-serif'>"
    + chip(f"{n_inst} instruments", T["blue"]) + chip(f"{len(cfg['strategies'])} strategies", T["green"])
    + chip(f"{len(cfg['portfolios'])} portfolios", T["amber"]) + chip(f"r = {R:.1%}", T["muted"])
    + chip(f"portfolio rule: {PORTFOLIO_RULE}", T["green"]) + f"&nbsp; config: <code>{CFG_SRC}</code></div>"))

KeyError: 'settings'

## Data basis: where each number comes from

Nothing here is measured from market data. A strategy only contributes its **stop distance**; the rest is instrument data from the YAML. Fields that are `null` show up as `verify`.

In [ ]:
STATUS_COL = {"backtested": T["green"], "assumed": T["amber"], "planned": T["red"]}
 
def basis_table():
    rows, accents = [], []
    for s, sd in cfg["strategies"].items():
        st = sd["stop"]
        rule = f"{st['multiple']:g} × ATR% × price" if st["type"] == "atr_multiple" else "fixed stop"
        origin = sd.get("source") or ("ATR% typed per instrument in YAML" if st["type"] == "atr_multiple" else "value typed in YAML")
        nights = sd.get("holding_nights")
        rows.append({"Strategy": s, "Status": sd.get("status", "unmarked"), "Stop rule": rule, "Stop comes from": origin,
                     "Gap control": "yes" if "gap_control" in sd else "no",
                     "Nights held": "verify" if nights is None else f"{nights:g}"})
        accents.append(SCOL[s])
    df = pd.DataFrame(rows)
    def paint(d):
        out = pd.DataFrame("", index=d.index, columns=d.columns)
        for i, v in d["Status"].items():
            out.loc[i, "Status"] = f"color: {STATUS_COL.get(v, T['muted'])}; font-weight: 600"
        return out
    show(df, "Strategies", text_cols=4, accents=accents, cell_css=paint)
 
def missing_inputs():
    used = sorted({(p, s) for st in cfg["strategies"].values() for p, syms in st["candidates"].items() for s in syms})
    rows = []
    for p, s in used:
        d = cfg["instruments"][p][s]
        miss = [k for k in ("commission_per_side", "swap_pct_annual", "spread_units") if d.get(k) is None]
        if miss:
            rows.append({"Instrument": f"{PL[p]['label']} · {s}", "Cost inputs not loaded": ", ".join(miss)})
    if rows:
        show(pd.DataFrame(rows), "Missing cost inputs (cost per trade shows `verify` or `>=`)", text_cols=2,
             accents=[PCOL[p] for p, s in used if [k for k in ("commission_per_side", "swap_pct_annual", "spread_units") if cfg["instruments"][p][s].get(k) is None]])
 
basis_table()
if any("status" not in sd for sd in cfg["strategies"].values()):
    display(Markdown("`unmarked` → add `status: backtested | assumed | planned` (and optionally `source:`) to that strategy in the YAML."))
missing_inputs()

## Strategy charts: minimum capital per candidate instrument

Bars are coloured by platform; the highlighted bar with **★** is the cheapest instrument on each platform. When a strategy has gap control, both scenarios sit side by side on the same scale.

In [ ]:
def plot_strategy(strategy):
    scs = scenarios_for(strategy)
    base = candidates(strategy, scs[-1]["gap_control"])          # fixed row order = strictest scenario
    n = len(base)
    labels = [f"{PL[r.plat]['label']} · {r.sym}" for r in base.itertuples()]
    fig, axes = plt.subplots(1, len(scs), figsize=(5.8 * len(scs) + 1.4, 0.46 * n + 1.9), sharex=True, sharey=True, squeeze=False)
    all_caps = [[min_capital(strategy, r.plat, r.sym, sc["gap_control"]) for r in base.itertuples()] for sc in scs]
    xmax = max(max(c) for c in all_caps)
    for ax, sc, caps in zip(axes[0], scs, all_caps):
        best = {p: v[0] for p, v in cheapest(strategy, sc["gap_control"]).items()}
        for i, (r, c) in enumerate(zip(base.itertuples(), caps)):
            is_best = r.sym == best[r.plat]
            ax.barh(i, c, height=0.58, color=to_rgba(PCOL[r.plat], 1.0 if is_best else 0.38))
            ax.text(c, i, esc(f"  {usd(c)}" + ("  ★" if is_best else "")), va="center", fontsize=9,
                    color=T["amber"] if is_best else T["muted"], fontweight="bold" if is_best else "normal")
        ax.set_yticks(range(n)); ax.set_yticklabels(labels)
        ax.set_xlim(0, xmax * 1.28)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v:,.0f}"))
        ax.set_title(sc["name"] if len(scs) > 1 else f"{strategy} · minimum capital (r = {R:.1%})")
    if len(scs) > 1:
        fig.suptitle(f"{strategy} · minimum capital (r = {R:.1%})", x=0.012, ha="left", fontsize=11, fontweight="bold", color=T["text"])
    axes[0][0].invert_yaxis()                                   # y is shared: invert once
    axes[0][-1].legend(handles=[Patch(color=PCOL[p], label=PL[p]["label"]) for p in PL], loc="lower right",
                       bbox_to_anchor=(1, 1.0), ncol=len(PL))
    fig.tight_layout(rect=(0, 0, 1, 0.94 if len(scs) > 1 else 1))
    plt.show()

for strat in cfg["strategies"]:
    plot_strategy(strat)

## Strategy tables: capital, margin, cost per trade and real risk

One table per strategy; the cheapest instrument on each platform is highlighted. `2x` / `5x` are multiples of the strictest scenario. Cost per trade only adds the known pieces (commission, swap over `holding_nights`, spread): a leading `>=` means the spread is not loaded, `verify` means a field is `null`.

In [ ]:
def trade_cost(strategy, plat, sym):
    d, nights = inst(plat, sym), cfg["strategies"][strategy].get("holding_nights")
    cps = d.get("commission_per_side")
    comm = NAN if cps is None else 2 * cps * d["usd_per_unit"]
    swp = d.get("swap_pct_annual")
    swap = NAN if (swp is None or nights is None) else d["notional"] * swp / 100 / 365 * nights
    sp = d.get("spread_units")
    spread = (sp if isinstance(sp, (int, float)) else 0) * d["usd_per_unit"]
    return comm, swap, spread
 
def strategy_table(strategy):
    scs = scenarios_for(strategy)
    multi = len(scs) > 1
    base = candidates(strategy, scs[-1]["gap_control"])
    best = {p: v[0] for p, v in cheapest(strategy, scs[-1]["gap_control"]).items()}
    extra_m = [m for m in S["multiples"] if m != 1]
    rows, accents, bold = [], [], []
    for r in base.itertuples():
        d = inst(r.plat, r.sym)
        comm, swap, spread = trade_cost(strategy, r.plat, r.sym)
        known = NAN if math.isnan(comm) or math.isnan(swap) else comm + swap + spread
        cost = "verify" if math.isnan(known) else (">= " if spread == 0 else "") + usd(known, 2)
        row = {"Instrument": f"{PL[r.plat]['label']} · {r.sym}", "Stop": r.stop, "Risk / lot": r.stop * d["usd_per_unit"]}
        caps = [min_capital(strategy, r.plat, r.sym, sc["gap_control"]) for sc in scs]
        for sc, c in zip(scs, caps):
            row[f"Min capital · {sc['name']}" if multi else "Min capital"] = c
        for m in extra_m:
            row[f"{m}x"] = caps[-1] * m
        row["Margin / lot"] = r.margin
        row["Cost / trade"] = cost
        rows.append(row); accents.append(PCOL[r.plat]); bold.append(r.sym == best[r.plat])
    df = pd.DataFrame(rows)
    money = {c: (lambda v: usd(v)) for c in df.columns if c.startswith("Min capital") or c.endswith("x")}
    money["Risk / lot"] = lambda v: usd(v, 1)
    money["Margin / lot"] = lambda v: usd(v, 1)
    money["Stop"] = lambda v: f"{v:g}"
    show(df, strategy, accents=accents, bold_rows=bold, fmt=money)
 
def risk_table(strategy):
    rows, accents = [], []
    for r in candidates(strategy).itertuples():
        risk = r.stop * inst(r.plat, r.sym)["usd_per_unit"]
        rows.append({"Instrument": f"{PL[r.plat]['label']} · {r.sym}", **{usd(e): risk / e for e in S["equity_levels"]}})
        accents.append(PCOL[r.plat])
    df = pd.DataFrame(rows)
    def heat(d):
        out = pd.DataFrame("", index=d.index, columns=d.columns)
        for c in d.columns[1:]:
            for i, v in d[c].items():
                if v <= R * (1 + 1e-9):   bg, fg = "rgba(52,211,153,.14)", T["green"]
                elif v <= 2 * R:          bg, fg = "rgba(245,165,36,.14)", T["amber"]
                else:                     bg, fg = "rgba(248,113,113,.14)", T["red"]
                out.loc[i, c] = f"background: {bg}; color: {fg}; font-weight: 600"
        return out
    pct = lambda v: f"{v:.2%}" if v < 0.001 else f"{v:.1%}"
    show(df, f"{strategy} · real risk of 1 minimum lot by equity   (green ≤ r · amber ≤ 2r · red > 2r, r = {R:.1%})",
         accents=accents, cell_css=heat, fmt={c: pct for c in df.columns[1:]})
 
for strat in cfg["strategies"]:
    strategy_table(strat)
    risk_table(strat)

## Portfolio charts: minimum capital with the cheapest instrument of each strategy

Each portfolio uses, per platform, the cheapest instrument of each strategy. The **stacked bar is the sum** (all strategies open at once, combined min-lot risk ≤ r); the **white tick marks the max** (each strategy alone ≤ r). `PORTFOLIO_RULE` decides which one feeds the `2x` / `5x` columns of the table.

In [ ]:
def plot_portfolio(pname):
    strategies = cfg["portfolios"][pname]
    has_gc, scs = scenarios_for_portfolio(pname)
    summs = [portfolio_summary(pname, sc["gap_control"]) for sc in scs]
    if not summs[0]:
        return
    n_plat = max(len(s) for s in summs)
    fig, axes = plt.subplots(len(scs), 1, figsize=(10.5, 0.95 * n_plat * len(scs) + 0.8 * len(scs) + 1.3), sharex=True, squeeze=False)
    xmax = max(v["cap_sum"] for s in summs for v in s.values())
    single = len(strategies) == 1
    for ax, sc, summ in zip(axes[:, 0], scs, summs):
        for i, (p, v) in enumerate(summ.items()):
            left = 0
            for strat in strategies:
                w = v["caps"][strat]
                ax.barh(i, w, left=left, height=0.55, color=SCOL[strat], edgecolor=T["panel"], linewidth=1.6)
                if w > xmax * 0.14:
                    ax.text(left + w / 2, i, esc(f"{strat}\n{usd(w)}"), ha="center", va="center", fontsize=7.5,
                            color=ink(SCOL[strat]), fontweight="bold")
                left += w
            if single:
                txt = f"  {usd(v['cap_sum'])}"
            else:
                ax.plot([v["cap_max"]] * 2, [i - 0.40, i + 0.40], color=T["text"], lw=1.8)
                txt = f"  Σ {usd(v['cap_sum'])}  ·  max {usd(v['cap_max'])}"
            ax.text(left, i, esc(txt), va="center", fontsize=9, color=T["text"], fontweight="bold")
        ax.set_yticks(range(len(summ))); ax.set_yticklabels([PL[p]["label"] for p in summ])
        ax.invert_yaxis(); ax.set_xlim(0, xmax * 1.36)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x:,.0f}"))
        if has_gc:
            ax.set_title(sc["name"])
    handles = [Patch(color=SCOL[s], label=s) for s in strategies]
    if not single:
        handles.append(Line2D([0], [0], color=T["text"], lw=1.8, label="max (each strategy alone)"))
    axes[0, 0].legend(handles=handles, loc="lower right", bbox_to_anchor=(1, 1.0), ncol=min(len(handles), 4))
    fig.suptitle(f"Portfolio '{pname}' · minimum capital (r = {R:.1%})", x=0.012, ha="left", fontsize=11, fontweight="bold", color=T["text"])
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()

def portfolio_table():
    extra_m = [m for m in S["multiples"] if m != 1]
    rows, accents, bold = [], [], []
    for pname, strategies in cfg["portfolios"].items():
        has_gc, scs = scenarios_for_portfolio(pname)
        for sc in scs:
            summ = portfolio_summary(pname, sc["gap_control"])
            if not summ:
                continue
            best_p = min(summ, key=lambda p: rule_cap(summ[p]))
            for p, v in sorted(summ.items(), key=lambda kv: rule_cap(kv[1])):
                row = {"Portfolio": pname, "Scenario": sc["name"] if has_gc else "—", "Platform": PL[p]["label"],
                       "Instruments": " · ".join(x[0] for x in v["picks"].values()),
                       "Capital (sum)": v["cap_sum"], "Capital (max)": v["cap_max"]}
                for m in extra_m:
                    row[f"{m}x ({PORTFOLIO_RULE})"] = rule_cap(v) * m
                row["Margin (min lots)"] = v["margin"]
                rows.append(row); accents.append(PCOL[p]); bold.append(p == best_p)
    df = pd.DataFrame(rows)
    def paint(d):
        out = pd.DataFrame("", index=d.index, columns=d.columns)
        col = "Capital (sum)" if PORTFOLIO_RULE == "sum" else "Capital (max)"
        out[col] = f"color: {T['green']}; font-weight: 700"
        return out
    fmt = {c: (lambda v: usd(v)) for c in df.columns if c.startswith("Capital") or c.endswith(f"({PORTFOLIO_RULE})")}
    fmt["Margin (min lots)"] = lambda v: usd(v, 1)
    comp = "  ·  ".join(f"**{p}** = {' + '.join(s)}" for p, s in cfg["portfolios"].items())
    display(Markdown(comp + f"\n\nHighlighted row = cheapest platform under the `{PORTFOLIO_RULE}` rule. **sum** = all open at once, **max** = each alone."))
    show(df, "Portfolio minimum capital by platform", text_cols=4, accents=accents, bold_rows=bold, cell_css=paint, fmt=fmt)

for pname in cfg["portfolios"]:
    plot_portfolio(pname)
portfolio_table()